# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review available record sets and their field `@id`s. All references to entities use their full `@id` as required for Croissant.

In [ ]:
# Display record sets with their @id and contained field/column ids
record_sets = list(dataset.record_sets)
print(f"\nAvailable record sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print(f"   Fields:")
        for f in fields:
            if isinstance(f, dict) and '@id' in f:
                print(f"     - {f['@id']}")
            elif isinstance(f, str):
                print(f"     - {f}")
    elif 'column' in rs:
        columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
        print(f"   Columns:")
        for c in columns:
            if isinstance(c, dict) and '@id' in c:
                print(f"     - {c['@id']}")
            elif isinstance(c, str):
                print(f"     - {c}")
    else:
        print("   No fields or columns found.")

## 3. Data Extraction
Load data from each record set into DataFrames using `mlcroissant`. Reference all sets and fields using their exact `@id`.

In [ ]:
# List all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        dataframes[rs_id] = pd.DataFrame(records)

# Show available DataFrames and columns
for rs_id, df in dataframes.items():
    print(f"Record set @id: {rs_id}")
    print(f"Columns (@id): {df.columns.tolist()}")
    print(df.head())
    print('-'*35)

# Select one (likely tabular) record set as main for further EDA
main_record_set_id = next(iter(dataframes.keys())) if dataframes else None
main_df = dataframes.get(main_record_set_id)
if main_df is not None:
    print(f"\nPreviewing first 5 rows from record set {main_record_set_id}:")
    display(main_df.head())
else:
    print("No tabular record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, and grouping. Use field `@id`s for selecting fields.

In [ ]:
import numpy as np

# We'll try to find a numeric column for demonstration by simple dtype inference
numeric_field_id = None
if main_df is not None:
    # Try to guess which column(s) are numeric: look for columns that can be converted to numeric
    for col in main_df.columns:
        try:
            vals = pd.to_numeric(main_df[col], errors='coerce')
            if vals.notnull().sum() > main_df.shape[0]//2:
                numeric_field_id = col
                main_df[col] = vals # force as numeric
                break
        except Exception:
            continue

if numeric_field_id:
    threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].notnull().any() else 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (using field @id):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field: choose first object-type column that's not the id
    group_field_id = None
    for col in main_df.columns:
        if main_df[col].dtype == object and col != numeric_field_id:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id, dropna=False)[numeric_field_id].mean()
        print(f"\nGrouped data by {group_field_id} (using field @id):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. At this stage, use field `@id`s for axis labels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example visualization: histogram and boxplot of the numeric field
if main_df is not None and numeric_field_id:
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')

    plt.subplot(1,2,2)
    sns.boxplot(x=main_df[numeric_field_id].dropna())
    plt.title(f'Boxplot of {numeric_field_id}')
    plt.xlabel(numeric_field_id)

    plt.tight_layout()
    plt.show()

    # If we found a group_field_id above, plot groupwise means
    if group_field_id:
        plt.figure(figsize=(8,5))
        group_means = main_df.groupby(group_field_id, dropna=False)[numeric_field_id].mean().sort_values(ascending=False)
        sns.barplot(y=group_means.index.astype(str), x=group_means.values)
        plt.title(f'Mean of {numeric_field_id} by {group_field_id}')
        plt.xlabel(numeric_field_id)
        plt.ylabel(group_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² dataset using `mlcroissant`, referencing all entities by their `@id`. We previewed tabular records, performed basic filtering and normalization on a numeric field, grouped the data by a categorical field using its `@id`, and visualized distributions.

**Key Observations:**
- The dataset schema and data access is controlled and documented by machine-readable Croissant `@id`s.
- Data exploration and transformation can be performed in a standard fashion using familiar data science tools, after retrieval via `mlcroissant`.
- Further analyses can be pursued as needed for clinical, statistical, or ML model research based on the rich, FAIR-compliant metadata.